# Июнь: какая таблица точнее покрывает `commission_monthly`

Эталон — отчёт эквайринга `06_Июнь_2026.xlsx`, колонка **«Комиссия (₽ в месяц)»**.

Сравниваем два источника на тех же ключах `ИНН + agr_id`:

| Источник | Поле месячной |
|---|---|
| **MPOS_RENT** (Альфа) | `n_amt` |
| **DOG_OPER** (ЦФТ) | `o.c_calc_summ` (все виды, без фильтра `commis_type`) |

## Два вопроса
1. **Сходится** с отчётом: одинаковое присутствие (оба ≠ 0 или оба пусто/0) и, если оба ≠ 0, совпадение суммы.
2. **Не сходится:**
   - заполнено в источнике, в отчёте 0;
   - заполнено в отчёте, в источнике нет строки или 0.

В конце — **VERDICT**, какая таблица ближе к отчёту как источник `commission_monthly`.

Новый kernel нормален. Нужны Impala и Excel на `/home/jovyan/documents/Equaring/Data`.  
Если уже гоняли `qc_june_mpos_excel_cft_monthly`, кадры `ex` / `mpos_map` / `cft` переиспользуются.


In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 140)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
OUT_DIR = DATA_DIR / 'qc_june_commission_monthly_winner'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MONTH = '2026-06'
MONTH_START = '2026-06-01'
MONTH_END = '2026-06-30'
MONTH_END_EXCL = '2026-07-01'
EXCEL_PATH = DATA_DIR / '06_Июнь_2026.xlsx'
EXCEL_HEADER = 0
VAT = 1.22
EPS = 0.01
MEM_LIMIT = '8g'
REUSE_IF_PRESENT = True
NOTEBOOK_REV = '2026-09-17-winner-v1'

print('NOTEBOOK_REV:', NOTEBOOK_REV)
print('MONTH:', MONTH)
print('EXCEL:', EXCEL_PATH, 'exists=', EXCEL_PATH.exists())
print('OUT_DIR:', OUT_DIR)
print('Эталон = отчёт июня | MPOS = n_amt | ЦФТ = o.c_calc_summ')


## 0) Helpers + Impala


In [ ]:
def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def to_num(s):
    return pd.to_numeric(
        s.astype(str).str.replace('\xa0', '', regex=False).str.replace(' ', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce',
    )


def pick_col(columns, aliases):
    cols = list(columns)
    norm = lambda x: re.sub(r'\s+', ' ', str(x).replace('\xa0', ' ').replace('\n', ' ').strip().lower())
    nmap = {norm(c): c for c in cols}
    for a in aliases:
        if a in cols:
            return a
        if norm(a) in nmap:
            return nmap[norm(a)]
    for a in aliases:
        key = norm(a)
        for nk, orig in nmap.items():
            if key and key in nk:
                return orig
    return None


def fetch_imp(sql, label):
    t0 = pd.Timestamp.now()
    print(f'FETCH {label} ...')
    with imp:
        imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
        df = imp.fetch(sql)
    if df is None:
        df = pd.DataFrame()
    print(f'  rows={len(df):,}  {(pd.Timestamp.now() - t0).total_seconds():.1f}s')
    return df


def is_nz(s):
    return pd.to_numeric(s, errors='coerce').fillna(0).abs() >= EPS


def _alive(name, need_cols):
    if not REUSE_IF_PRESENT or name not in globals():
        return False
    obj = globals()[name]
    return isinstance(obj, pd.DataFrame) and len(obj) > 0 and all(c in obj.columns for c in need_cols)


if 'imp' in globals() and imp is not None:
    print('Reuse existing imp')
else:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'},
    )
    imp._init_connection()
    print('Impala connected')


## 1) Эталон: отчёт июня


In [ ]:
if _alive('ex', ['inn_key', 'agr_id_key', 'commission_excel']):
    print('Reuse in-memory `ex`')
else:
    if not EXCEL_PATH.exists():
        raise FileNotFoundError(f'Нет файла отчёта: {EXCEL_PATH}')
    raw_ex = pd.read_excel(EXCEL_PATH, header=EXCEL_HEADER)
    print('Excel raw rows=', f'{len(raw_ex):,}', '| cols=', list(raw_ex.columns)[:25])
    col_inn = pick_col(raw_ex.columns, ['ИНН', 'inn', 'c_inn'])
    col_agr = pick_col(raw_ex.columns, ['ID договора', 'agr_id', 'abs_agr_id', 'ИД договора'])
    col_mon = pick_col(raw_ex.columns, [
        'Комиссия (₽ в месяц)', 'Комиссия (руб в месяц)', 'Комиссия в месяц',
        'Комиссия CN (₽ в месяц)', 'commission_monthly',
    ])
    col_tar = pick_col(raw_ex.columns, ['Тариф', 'Тарифный план', 'tariff_name'])
    print('resolved:', {'inn': col_inn, 'agr': col_agr, 'monthly': col_mon, 'tariff': col_tar})
    if None in (col_inn, col_agr, col_mon):
        raise RuntimeError(f'Не найдены колонки Excel. cols={list(raw_ex.columns)}')
    ex = pd.DataFrame({
        'inn_key': raw_ex[col_inn].map(normalize_inn_q1),
        'agr_id_key': raw_ex[col_agr].map(normalize_agr_q1),
        'commission_excel': to_num(raw_ex[col_mon]),
        'tariff': raw_ex[col_tar] if col_tar else np.nan,
    })
    ex = (
        ex.dropna(subset=['agr_id_key'])
        .groupby(['inn_key', 'agr_id_key'], as_index=False)
        .agg(commission_excel=('commission_excel', 'max'), tariff=('tariff', 'first'))
    )

ex = ex.copy()
ex['commission_excel'] = pd.to_numeric(ex['commission_excel'], errors='coerce').fillna(0)
ex['excel_nz'] = is_nz(ex['commission_excel'])
ex['excel_zero'] = ~ex['excel_nz']
n_ex = len(ex)
n_ex_nz = int(ex['excel_nz'].sum())
n_ex_z = int(ex['excel_zero'].sum())
print('=== Эталон Excel June ===')
print(f'keys={n_ex:,}  nonzero={n_ex_nz:,} ({100.0 * n_ex_nz / n_ex:.2f}%)  zero={n_ex_z:,} ({100.0 * n_ex_z / n_ex:.2f}%)')
print('sum nonzero=', f"{ex.loc[ex['excel_nz'], 'commission_excel'].sum():,.2f}")
display(ex.groupby('excel_nz', as_index=False).agg(keys=('agr_id_key', 'nunique'), inn=('inn_key', 'nunique'), sum_comm=('commission_excel', 'sum')))


## 2) MPOS_RENT → `n_amt` на inn+agr


In [ ]:
sql_mpos = f'''
with rent_base as (
  select
    cast(c_nmrc as string) as c_nmrc,
    cast(d_rent as date) as d_rent_dt,
    cast(n_amt as double) as n_amt_num,
    cast(ods_commit_ts as timestamp) as ods_commit_ts,
    cast(ods_insert_ts as timestamp) as ods_insert_ts,
    cast(ods_op_csn as decimal(38, 0)) as ods_op_csn,
    coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
  from ods_alpha.scd1_mrc_pos_rent
  where c_nmrc is not null
    and cast(d_rent as date) between cast('{MONTH_START}' as date) and cast('{MONTH_END}' as date)
),
rent_ranked as (
  select *,
    row_number() over (
      partition by c_nmrc, d_rent_dt
      order by coalesce(ods_commit_ts, ods_insert_ts) desc, ods_op_csn desc, ods_insert_ts desc
    ) as rn
  from rent_base
  where ods_deleted_flg not in ('1', 'Y', 'y')
),
rent_dedup as (
  select c_nmrc, d_rent_dt, n_amt_num from rent_ranked where rn = 1
),
terms_active as (
  select distinct
    cast(t.n_agr as string) as n_agr,
    cast(t.c_nmrc as string) as c_nmrc,
    cast(t.d_valid_from as date) as d_valid_from,
    cast(t.d_valid_to as date) as d_valid_to
  from ods_alpha.scd1_agr_terms t
  where coalesce(cast(t.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and t.c_nmrc is not null
    and cast(t.d_valid_from as date) <= cast('{MONTH_END}' as date)
    and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{MONTH_START}' as date))
),
agreements_active as (
  select distinct
    cast(a.n_agr as string) as n_agr,
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_cmp_client as string) as n_cmp_client,
    cast(a.d_valid_from as date) as d_valid_from,
    cast(a.d_valid_to as date) as d_valid_to
  from ods_alpha.scd1_agreements a
  where coalesce(cast(a.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and upper(trim(cast(a.acq_class as string))) = 'SA'
    and cast(a.d_valid_from as date) <= cast('{MONTH_END}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{MONTH_START}' as date))
),
companies_active as (
  select distinct
    cast(c.n_cmp as string) as n_cmp,
    regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_key
  from ods_alpha.scd1_companies c
  where coalesce(cast(c.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and c.c_inn is not null
)
select
  c.inn_key as inn,
  cast(a.agr_id as string) as agr_id,
  count(*) as rent_rows,
  sum(r.n_amt_num) as commission_mpos
from rent_dedup r
left join terms_active t
  on t.c_nmrc = r.c_nmrc
 and r.d_rent_dt between t.d_valid_from and coalesce(t.d_valid_to, cast('2999-12-31' as date))
left join agreements_active a
  on a.n_agr = t.n_agr
 and r.d_rent_dt between a.d_valid_from and coalesce(a.d_valid_to, cast('2999-12-31' as date))
left join companies_active c
  on c.n_cmp = a.n_cmp_client
group by 1, 2
'''

if _alive('mpos_map', ['inn_key', 'agr_id_key', 'commission_mpos']):
    print('Reuse in-memory `mpos_map`')
else:
    mpos_raw = fetch_imp(sql_mpos, 'MPOS mapped June')
    mpos = mpos_raw.copy()
    mpos['inn_key'] = mpos['inn'].map(normalize_inn_q1)
    mpos['agr_id_key'] = mpos['agr_id'].map(normalize_agr_q1)
    mpos['commission_mpos'] = pd.to_numeric(mpos['commission_mpos'], errors='coerce')
    mpos_map = (
        mpos.dropna(subset=['agr_id_key'])
        .groupby(['inn_key', 'agr_id_key'], as_index=False)
        .agg(commission_mpos=('commission_mpos', 'sum'), rent_rows=('rent_rows', 'sum'))
    )

mpos_map = mpos_map.copy()
mpos_map['commission_mpos'] = pd.to_numeric(mpos_map['commission_mpos'], errors='coerce')
mpos_map['mpos_nz'] = is_nz(mpos_map['commission_mpos'])
print('MPOS keys=', f'{len(mpos_map):,}', '| nonzero=', int(mpos_map['mpos_nz'].sum()))
print('sum mpos=', f"{mpos_map['commission_mpos'].fillna(0).sum():,.2f}")


## 3) ЦФТ: месячная = `o.c_calc_summ`

Все строки `DOG_OPER` за июнь, сумма на `inn+agr`. `commis_type` не фильтрует.


In [ ]:
sql_cft = f'''
select
  regexp_replace(trim(cast(cl.c_inn as string)), '[^0-9]', '') as inn,
  cast(m.id as string) as agr_id,
  cast(vc.id as string) as vid_id,
  cast(vc.c_name as string) as commis_type,
  cast(o.c_date_create as timestamp) as c_date_create,
  cast(o.c_pay_summ as double) as c_pay_summ,
  cast(o.c_calc_summ as double) as c_calc_summ
from ods.scd1_z_R2_IP_DOG_OPER o
join ods.scd1_z_R2_VID_COMISS vc on vc.id = o.c_vid_comiss
join ods.scd1_z_r2_ip_merchants m on m.id = o.c_parent_id
join ods.scd1_z_client cl on m.c_cl_org = cl.id
where o.c_parent_class = 'R2_IP_MERCHANTS'
  and cl.class_id = 'CL_ORG'
  and cast(o.c_date_create as date) >= cast('{MONTH_START}' as date)
  and cast(o.c_date_create as date) < cast('{MONTH_END_EXCL}' as date)
'''

if _alive('cft', ['c_calc_summ', 'agr_id']) or _alive('cft', ['c_calc_summ', 'agr_id_key']):
    print('Reuse in-memory `cft` — re-aggregate sum(c_calc_summ)')
    cft_raw = cft.copy()
else:
    try:
        cft_raw = fetch_imp(sql_cft, 'CFT DOG_OPER June')
    except Exception as exc:
        print('mixed-case fail, try lowercase:', type(exc).__name__, exc)
        sql_cft = sql_cft.replace('scd1_z_R2_IP_DOG_OPER', 'scd1_z_r2_ip_dog_oper').replace('scd1_z_R2_VID_COMISS', 'scd1_z_r2_vid_comiss')
        cft_raw = fetch_imp(sql_cft, 'CFT DOG_OPER June lc')

cft = cft_raw.copy()
if 'inn_key' not in cft.columns:
    cft['inn_key'] = cft['inn'].map(normalize_inn_q1)
if 'agr_id_key' not in cft.columns:
    cft['agr_id_key'] = cft['agr_id'].map(normalize_agr_q1)
cft['c_calc_summ'] = pd.to_numeric(cft['c_calc_summ'], errors='coerce')
cft['c_pay_summ'] = pd.to_numeric(cft.get('c_pay_summ'), errors='coerce')
cft = cft.dropna(subset=['agr_id_key'])

_cft_agg = {
    'cft_rows': ('c_calc_summ', 'size'),
    'commission_monthly_cft': ('c_calc_summ', 'sum'),
    'commission_cft_pay': ('c_pay_summ', 'sum'),
}
if 'commis_type' in cft.columns:
    _cft_agg['commis_types'] = ('commis_type', lambda s: ' | '.join(sorted({str(x) for x in s.dropna()})))
cft_map = cft.groupby(['inn_key', 'agr_id_key'], as_index=False).agg(**_cft_agg)
cft_map['cft_nz'] = is_nz(cft_map['commission_monthly_cft'])
print('CFT monthly = sum(c_calc_summ), no type filter')
print('CFT keys=', f'{len(cft_map):,}', '| rows=', f'{len(cft):,}', '| nonzero=', int(cft_map['cft_nz'].sum()))
print('sum c_calc_summ=', f"{cft_map['commission_monthly_cft'].fillna(0).sum():,.2f}")
print('sum c_pay_summ=', f"{cft_map['commission_cft_pay'].fillna(0).sum():,.2f}")


## 4) Счётчики: сходится / не сходится

На периметре отчёта (все ключи Excel июня).

| Бакет | Смысл |
|---|---|
| оба ≠ 0 | отчёт и источник заполнены |
| оба 0 / нет строки | отчёт 0 и в источнике нет ненуля (нет строки или 0) |
| источник ≠ 0, отчёт = 0 | ложное заполнение |
| отчёт ≠ 0, в источнике нет строки | пропуск |
| отчёт ≠ 0, источник = 0 | пропуск (строка есть, сумма 0) |

**Сходится (присутствие)** = оба ≠ 0 + оба 0.  
Сумма проверяется отдельно на пересечении ненулей: как есть и ×1.22 (НДС).


In [ ]:
def classify_row(excel_nz, src_present, src_nz):
    if excel_nz and src_nz:
        return 'оба ≠ 0'
    if (not excel_nz) and (not src_nz):
        return 'оба 0 / нет строки'
    if (not excel_nz) and src_nz:
        return 'источник ≠ 0, отчёт = 0'
    if excel_nz and not src_present:
        return 'отчёт ≠ 0, в источнике нет строки'
    return 'отчёт ≠ 0, источник = 0'


def score_source(name, src_df, src_col):
    t = ex.merge(
        src_df[['inn_key', 'agr_id_key', src_col]].copy(),
        on=['inn_key', 'agr_id_key'],
        how='left',
    )
    t['src_amt'] = pd.to_numeric(t[src_col], errors='coerce')
    t['src_present'] = t['src_amt'].notna()
    t['src_nz'] = is_nz(t['src_amt'])
    t['bucket'] = [
        classify_row(bool(ez), bool(sp), bool(sz))
        for ez, sp, sz in zip(t['excel_nz'], t['src_present'], t['src_nz'])
    ]
    t['excel_gross'] = t['commission_excel'] * VAT
    t['delta_net'] = t['src_amt'] - t['commission_excel']
    t['delta_vat'] = t['src_amt'] - t['excel_gross']
    both = t.loc[t['bucket'] == 'оба ≠ 0'].copy()
    if len(both):
        both['exact_net'] = np.isclose(both['src_amt'].fillna(0), both['commission_excel'].fillna(0), atol=EPS, rtol=0)
        both['exact_vat'] = np.isclose(both['src_amt'].fillna(0), both['excel_gross'].fillna(0), atol=EPS, rtol=0)
        both['near_net'] = np.isclose(both['src_amt'].fillna(0), both['commission_excel'].fillna(0), atol=EPS, rtol=0.01)
        both['near_vat'] = np.isclose(both['src_amt'].fillna(0), both['excel_gross'].fillna(0), atol=EPS, rtol=0.01)
        t = t.merge(both[['inn_key', 'agr_id_key', 'exact_net', 'exact_vat', 'near_net', 'near_vat']], on=['inn_key', 'agr_id_key'], how='left')
    else:
        t['exact_net'] = False
        t['exact_vat'] = False
        t['near_net'] = False
        t['near_vat'] = False

    n = len(t)
    vc = t['bucket'].value_counts()
    agree = int(vc.get('оба ≠ 0', 0) + vc.get('оба 0 / нет строки', 0))
    fp = t.loc[t['bucket'] == 'источник ≠ 0, отчёт = 0']
    fn_abs = t.loc[t['bucket'] == 'отчёт ≠ 0, в источнике нет строки']
    fn_z = t.loc[t['bucket'] == 'отчёт ≠ 0, источник = 0']
    fn = pd.concat([fn_abs, fn_z], ignore_index=True)

    rec = {
        'source': name,
        'n_excel': n,
        'excel_nz': int(t['excel_nz'].sum()),
        'excel_zero': int((~t['excel_nz']).sum()),
        'src_present': int(t['src_present'].sum()),
        'src_nz': int(t['src_nz'].sum()),
        'agree_n': agree,
        'agree_pct': round(100.0 * agree / n, 2) if n else np.nan,
        'both_nz': int(vc.get('оба ≠ 0', 0)),
        'both_empty': int(vc.get('оба 0 / нет строки', 0)),
        'fp_src_nz_excel_zero': int(len(fp)),
        'fp_pct_of_excel': round(100.0 * len(fp) / n, 2) if n else np.nan,
        'fp_pct_of_excel_zero': round(100.0 * len(fp) / n_ex_z, 2) if n_ex_z else np.nan,
        'fp_sum_src': float(fp['src_amt'].fillna(0).sum()),
        'fn_excel_nz_src_absent': int(len(fn_abs)),
        'fn_excel_nz_src_zero': int(len(fn_z)),
        'fn_total': int(len(fn)),
        'fn_pct_of_excel': round(100.0 * len(fn) / n, 2) if n else np.nan,
        'fn_pct_of_excel_nz': round(100.0 * len(fn) / n_ex_nz, 2) if n_ex_nz else np.nan,
        'fn_sum_excel': float(fn['commission_excel'].fillna(0).sum()),
        'recall_excel_nz': round(100.0 * int(vc.get('оба ≠ 0', 0)) / n_ex_nz, 2) if n_ex_nz else np.nan,
        'precision_src_nz': round(100.0 * int(vc.get('оба ≠ 0', 0)) / int(t['src_nz'].sum()), 2) if int(t['src_nz'].sum()) else np.nan,
        'exact_net_n': int(both['exact_net'].sum()) if len(both) else 0,
        'exact_net_pct': round(100.0 * both['exact_net'].mean(), 2) if len(both) else np.nan,
        'exact_vat_n': int(both['exact_vat'].sum()) if len(both) else 0,
        'exact_vat_pct': round(100.0 * both['exact_vat'].mean(), 2) if len(both) else np.nan,
        'near_net_pct': round(100.0 * both['near_net'].mean(), 2) if len(both) else np.nan,
        'near_vat_pct': round(100.0 * both['near_vat'].mean(), 2) if len(both) else np.nan,
        'sum_excel_both_nz': float(both['commission_excel'].sum()) if len(both) else 0.0,
        'sum_src_both_nz': float(both['src_amt'].sum()) if len(both) else 0.0,
        'sum_excel_nz': float(t.loc[t['excel_nz'], 'commission_excel'].sum()),
        'sum_src_on_excel': float(t['src_amt'].fillna(0).sum()),
    }
    best_amt = 'net' if (rec['exact_net_pct'] or 0) >= (rec['exact_vat_pct'] or 0) else 'vat_x1.22'
    rec['best_amount_scale'] = best_amt
    rec['best_amount_exact_pct'] = rec['exact_net_pct'] if best_amt == 'net' else rec['exact_vat_pct']
    rec['disagree_n'] = rec['fp_src_nz_excel_zero'] + rec['fn_total']
    rec['disagree_pct'] = round(100.0 * rec['disagree_n'] / n, 2) if n else np.nan
    return t, rec


mpos_tri, mpos_score = score_source('MPOS_RENT n_amt', mpos_map, 'commission_mpos')
cft_tri, cft_score = score_source('CFT o.c_calc_summ', cft_map, 'commission_monthly_cft')
scores = pd.DataFrame([mpos_score, cft_score])

bucket_order = [
    'оба ≠ 0',
    'оба 0 / нет строки',
    'источник ≠ 0, отчёт = 0',
    'отчёт ≠ 0, в источнике нет строки',
    'отчёт ≠ 0, источник = 0',
]


def bucket_table(tri, name):
    g = (
        tri.groupby('bucket', as_index=False)
        .agg(
            keys=('agr_id_key', 'nunique'),
            inn=('inn_key', 'nunique'),
            sum_excel=('commission_excel', 'sum'),
            sum_src=('src_amt', 'sum'),
        )
    )
    g['source'] = name
    g['pct_excel'] = (100.0 * g['keys'] / n_ex).round(2)
    g['bucket'] = pd.Categorical(g['bucket'], categories=bucket_order, ordered=True)
    return g.sort_values('bucket')


buck = pd.concat([
    bucket_table(mpos_tri, 'MPOS_RENT'),
    bucket_table(cft_tri, 'CFT c_calc_summ'),
], ignore_index=True)

print('=== 1) СХОДИТСЯ с отчётом июня ===')
show_agree = scores[[
    'source', 'agree_n', 'agree_pct', 'both_nz', 'both_empty',
    'recall_excel_nz', 'precision_src_nz',
    'exact_net_pct', 'exact_vat_pct', 'best_amount_scale', 'best_amount_exact_pct',
]].copy()
display(show_agree)

print('\\n=== 2) НЕ СХОДИТСЯ (обе стороны) ===')
show_bad = scores[[
    'source', 'disagree_n', 'disagree_pct',
    'fp_src_nz_excel_zero', 'fp_pct_of_excel', 'fp_pct_of_excel_zero', 'fp_sum_src',
    'fn_total', 'fn_excel_nz_src_absent', 'fn_excel_nz_src_zero',
    'fn_pct_of_excel_nz', 'fn_sum_excel',
]].copy()
display(show_bad)

print('\\n=== Бакеты на периметре отчёта ===')
display(buck)


## 5) VERDICT — какая таблица точнее


In [ ]:
def pick_winner(a, b, key, higher=True):
    va, vb = a[key], b[key]
    if pd.isna(va) and pd.isna(vb):
        return None
    if pd.isna(va):
        return b['source']
    if pd.isna(vb):
        return a['source']
    if va == vb:
        return None
    if higher:
        return a['source'] if va > vb else b['source']
    return a['source'] if va < vb else b['source']


w_agree = pick_winner(mpos_score, cft_score, 'agree_pct', higher=True)
w_fp = pick_winner(mpos_score, cft_score, 'fp_pct_of_excel', higher=False)
w_fn = pick_winner(mpos_score, cft_score, 'fn_pct_of_excel_nz', higher=False)
w_amt = pick_winner(mpos_score, cft_score, 'best_amount_exact_pct', higher=True)

# Итог: сначала присутствие как в отчёте, штраф за ложные заполнения и пропуски, затем сумма.
mpos_pts = 0
cft_pts = 0
for w in (w_agree, w_fp, w_fn, w_amt):
    if w and w.startswith('MPOS'):
        mpos_pts += 1
    elif w and w.startswith('CFT'):
        cft_pts += 1

if mpos_pts > cft_pts:
    winner = 'MPOS_RENT (`n_amt`)'
elif cft_pts > mpos_pts:
    winner = 'ЦФТ DOG_OPER (`o.c_calc_summ`)'
else:
    winner = 'НИЧЬЯ — смотри разрез присутствие vs сумма'

print('=' * 72)
print('VERDICT: покрытие commission_monthly относительно отчёта июня')
print('=' * 72)
print(f'Периметр отчёта: {n_ex:,} договоров | ≠0 = {n_ex_nz:,} ({100.0 * n_ex_nz / n_ex:.2f}%) | =0 = {n_ex_z:,}')
print()
print('1) СХОДИТСЯ (присутствие = как в отчёте)')
for rec in (mpos_score, cft_score):
    print(
        f"   {rec['source']}: {rec['agree_n']:,}/{rec['n_excel']:,} = {rec['agree_pct']:.2f}%"
        f"  | оба≠0 {rec['both_nz']:,} | оба пусто {rec['both_empty']:,}"
        f"  | recall ненулей отчёта {rec['recall_excel_nz']:.2f}%"
        f"  | precision {rec['precision_src_nz']:.2f}%"
    )
print(f'   лучше по присутствию: {w_agree or "одинаково"}')
print()
print('   Сумма на пересечении оба≠0:')
for rec in (mpos_score, cft_score):
    print(
        f"   {rec['source']}: exact vs Excel {rec['exact_net_pct']}% |"
        f" exact vs Excel×{VAT} {rec['exact_vat_pct']}% |"
        f" лучше шкала {rec['best_amount_scale']} → {rec['best_amount_exact_pct']}%"
        f"  | sum Excel {rec['sum_excel_both_nz']:,.2f} vs src {rec['sum_src_both_nz']:,.2f}"
    )
print(f'   лучше по сумме: {w_amt or "одинаково"}')
print()
print('2) НЕ СХОДИТСЯ')
print('   а) заполнено в источнике, в отчёте 0')
for rec in (mpos_score, cft_score):
    print(
        f"   {rec['source']}: {rec['fp_src_nz_excel_zero']:,} ключей"
        f" ({rec['fp_pct_of_excel']:.2f}% отчёта, {rec['fp_pct_of_excel_zero']:.2f}% нулей отчёта)"
        f"  сумма источника {rec['fp_sum_src']:,.2f}"
    )
print(f'   меньше ложных заполнений: {w_fp or "одинаково"}')
print()
print('   б) заполнено в отчёте, в источнике нет / 0')
for rec in (mpos_score, cft_score):
    print(
        f"   {rec['source']}: {rec['fn_total']:,} ключей"
        f" (нет строки {rec['fn_excel_nz_src_absent']:,} + ноль {rec['fn_excel_nz_src_zero']:,})"
        f" = {rec['fn_pct_of_excel_nz']:.2f}% ненулей отчёта, сумма отчёта {rec['fn_sum_excel']:,.2f}"
    )
print(f'   меньше пропусков: {w_fn or "одинаково"}')
print()
print('-' * 72)
print(f'ИТОГ: точнее для commission_monthly → {winner}')
print(f'Очки (присутствие / меньше FP / меньше FN / сумма)  MPOS {mpos_pts} : ЦФТ {cft_pts}')
print()
if mpos_score['agree_pct'] >= cft_score['agree_pct'] and mpos_score['fp_pct_of_excel'] <= cft_score['fp_pct_of_excel']:
    print('Смысл: MPOS ближе к флагу месячной в отчёте (нули в Альфу почти не пишутся).')
elif cft_score['recall_excel_nz'] > mpos_score['recall_excel_nz'] and cft_score['fp_pct_of_excel'] > mpos_score['fp_pct_of_excel']:
    print('Смысл: ЦФТ ловит больше ненулей отчёта, но шире — много заполнений там, где отчёт 0.')
print('=' * 72)

# Топы расхождений
print('\\n=== TOP: источник ≠ 0, отчёт = 0 ===')
for name, tri in [('MPOS', mpos_tri), ('CFT', cft_tri)]:
    top = (
        tri.loc[tri['bucket'] == 'источник ≠ 0, отчёт = 0']
        .assign(abs_src=lambda d: d['src_amt'].abs())
        .sort_values('abs_src', ascending=False)
        [['inn_key', 'agr_id_key', 'tariff', 'commission_excel', 'src_amt']]
        .head(15)
    )
    print(name)
    display(top)

print('\\n=== TOP: отчёт ≠ 0, источник пуст/0 ===')
for name, tri in [('MPOS', mpos_tri), ('CFT', cft_tri)]:
    top = (
        tri.loc[tri['bucket'].isin(['отчёт ≠ 0, в источнике нет строки', 'отчёт ≠ 0, источник = 0'])]
        .sort_values('commission_excel', ascending=False)
        [['inn_key', 'agr_id_key', 'tariff', 'commission_excel', 'src_amt', 'bucket']]
        .head(15)
    )
    print(name)
    display(top)


## 6) Выгрузка


In [ ]:
out = OUT_DIR / f'june_commission_monthly_winner_{MONTH}.xlsx'
mpos_fp = mpos_tri.loc[mpos_tri['bucket'] == 'источник ≠ 0, отчёт = 0'].copy()
mpos_fn = mpos_tri.loc[mpos_tri['bucket'].isin(['отчёт ≠ 0, в источнике нет строки', 'отчёт ≠ 0, источник = 0'])].copy()
cft_fp = cft_tri.loc[cft_tri['bucket'] == 'источник ≠ 0, отчёт = 0'].copy()
cft_fn = cft_tri.loc[cft_tri['bucket'].isin(['отчёт ≠ 0, в источнике нет строки', 'отчёт ≠ 0, источник = 0'])].copy()

with pd.ExcelWriter(out, engine='openpyxl') as w:
    scores.to_excel(w, sheet_name='scores', index=False)
    buck.to_excel(w, sheet_name='buckets', index=False)
    mpos_tri.to_excel(w, sheet_name='mpos_vs_excel', index=False)
    cft_tri.to_excel(w, sheet_name='cft_vs_excel', index=False)
    mpos_fp.to_excel(w, sheet_name='mpos_src_nz_excel_0', index=False)
    mpos_fn.to_excel(w, sheet_name='mpos_excel_nz_src_empty', index=False)
    cft_fp.to_excel(w, sheet_name='cft_src_nz_excel_0', index=False)
    cft_fn.to_excel(w, sheet_name='cft_excel_nz_src_empty', index=False)
print('Saved:', out)
print('Готово. Заключение — ячейка VERDICT выше.')
